This notebook is meant to be run on the HPC via Europa. Instructions are here:
* https://github.com/NatLabRockies/HPC/tree/master/general/Jupyterhub/jupyter

In [ ]:
import numpy as np
# import gdxpds
import pandas as pd
import matplotlib as mpl
import matplotlib.pyplot as plt
import os, sys, math, site, shutil, importlib
from tqdm import tqdm, trange
from glob import glob

import geopandas as gpd
import h5py

# ### ReEDS: SA_NTPS branch
# reedspath = os.path.expanduser('~/github/ReEDS/')
# reedspath2 = os.path.expanduser('~/github2/ReEDS/')
# remotepath = '/Volumes/ReEDS/'
# projpath = os.path.expanduser('~/Projects/SpatialTemporal/')


# reedspath = reedspath2


# site.addsitedir(os.path.join(reedspath,'postprocessing'))
# import plots
# import reedsplots as rplots
# importlib.reload(rplots)

pd.options.display.max_columns = 200

# CSP profile files

In [ ]:
inpath = '/shared-projects/rev/projects/csp_conus/csp/'
year = 2012
infile = os.path.join(inpath,f'csp_gen_{year}.h5')

with h5py.File(infile, 'r') as f:
    keys = list(f)
    for key in keys:
        print(key)
#         display(pd.DataFrame(f[key]))

    meta = pd.DataFrame(f['meta'][...])
    meta['cf_mean'] = pd.DataFrame(f['cf_mean'][...])
    meta['dni_mean'] = pd.DataFrame(f['dni_mean'][...])
    meta['lcoe_real'] = pd.DataFrame(f['lcoe_real'][...])
    time_index = pd.DataFrame(f['time_index'][...])

for c in ['country','state','county','urban','reV_tech']:
    meta[c] = meta[c].str.decode('UTF-8')

time_index = time_index[0].str.decode('UTF-8')[::2]

In [ ]:
# os.makedirs(os.path.expanduser('~/reV'), exist_ok=True)
# meta.to_hdf(
#     os.path.expanduser('~/reV/csp_meta.h5'),
#     key='data', complevel=4)

In [ ]:
# meta.to_csv(os.path.expanduser('~/reV/csp/csp_meta.csv.gz'), index=False)

In [ ]:
meta

In [ ]:
print(meta.cf_mean.describe())
dfplot = meta.loc[meta.cf_mean>=156].copy()

In [ ]:
plt.close()
f,ax = plt.subplots(figsize=(13,8))
ax.scatter(
    dfplot.longitude, dfplot.latitude, c=dfplot.dni_mean,
    s=0.6, lw=0, marker='s', cmap=plt.cm.gist_earth_r, vmin=0,
)
plt.show()

In [ ]:
with h5py.File(infile, 'r') as f:
    cf_profile = pd.DataFrame(f['cf_profile'][:,:5])

In [ ]:
cf_profile.iloc[:24*2*7].plot()

In [ ]:
plt.close()
f,ax = plt.subplots()
cf_profile[0].sort_values().reset_index(drop=True).plot(ax=ax)
ax.set_yscale('log')
plt.show()

In [ ]:
cf_profile.describe()

In [ ]:
df = cf_profile.astype(float)/1000
df[0].loc[df[0] < 0.01] = 0

In [ ]:
plt.close()
f,ax = plt.subplots()
df[0].sort_values().reset_index(drop=True).plot(ax=ax)
ax.set_yscale('log')
plt.show()

In [ ]:
df[0].replace(0,np.nan).dropna().sort_values()

In [ ]:
with h5py.File(infile, 'r') as f:
    cf_hour0 = pd.DataFrame(f['cf_profile'][0,:])

# Assemble profiles

In [ ]:
inpath = '/shared-projects/rev/projects/csp_conus/csp/'
years = list(range(2007, 2014))

# cffile = 
# with h5py.File(infile, 'r') as f:
#     cf_profile = pd.DataFrame(f['cf_profile'][:,:5])

In [ ]:
### Get supply curve
dfsc = gpd.read_file(
    os.path.expanduser('~/reV/csp/vision_sn2_csp_conus_2012-sc_point_gid.gpkg')
)

In [ ]:
dfsc.groupby('sc_point_gid').profile_index.count()

In [ ]:
dfsc.groupby('sc_point_gid').profile_index.unique().map(lambda x: len(x)).sort_values()

In [ ]:
sc_point_gids = sorted(dfsc.sc_point_gid.unique())
print(len(sc_point_gids))

In [ ]:
timeindex = {
    y: pd.date_range(f'{y}-01-01', f'{y+1}-01-01', closed='left', freq='H', tz='EST')[:8760]
    for y in range(2007,2014)
}

fulltimeindex = pd.concat([pd.Series(index=t, dtype=float) for t in timeindex.values()]).index
fulltimeindex

In [ ]:
timeindex[2012]

In [ ]:
timeindex[2012].tz_convert('EST')

In [ ]:
### EST is 5 hours from UTC

In [ ]:
cfout = {}
# for sc_point_gid in [97462]:
for sc_point_gid in tqdm(sc_point_gids):
    scpoints = dfsc.loc[dfsc.sc_point_gid==sc_point_gid].copy()

    mwsite = scpoints.capacity_mw.sum()
    profile_indices = scpoints.profile_index.unique()

    ### Get the CF profiles
    cfin = {}
    for year in years:
        infile = os.path.join(inpath,f'csp_gen_{year}.h5')
        with h5py.File(infile, 'r') as f:
            ## Keep instantaneous values on the hour (::2)
            cfin[year] = pd.DataFrame(f['cf_profile'][::2, profile_indices])
            ## Convert from UTC to EST
            cfin[year].loc[:,:] = np.roll(cfin[year], -5, axis=0)

    ## Concat into one dataframe
    cfallyears = pd.concat(cfin, axis=0, ignore_index=True)
    cfallyears.columns = profile_indices
    ## Clip the small values
    for c in cfallyears:
        cfallyears.loc[cfallyears[c] < 10, c] = 0
    ## Convert to fractional cf
    cfallyears = cfallyears / 1000

    ### Get available-capacity-weighted average for sc_point_gid
    cfrows = {}
    for i, row in scpoints.iterrows():
        cfrows[i] = cfallyears[row.profile_index] * row.capacity_mw

    cfsite = (pd.concat(cfrows, axis=1).sum(axis=1) / mwsite).astype(np.float32)

#     ### Convert timezone
#     dfsite.index = 
    
    ### Store it
    cfout[sc_point_gid] = cfsite

In [ ]:
dfwrite = pd.concat(cfout, axis=1)
dfwrite.index = fulltimeindex

In [ ]:
# dfwrite.to_csv('cspcf_backup.csv')

dfwrite.to_hdf('cspcf_est.h5', key='data', complevel=4)

In [ ]:
foo = cfin[year].copy()
foo.loc[:,:] = np.roll(foo, -5, axis=0)

foo.loc[24*100:24*101].plot()